In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET s3_endpoint='minio:9000';")
con.execute("SET s3_use_ssl=false;")
con.execute("SET s3_url_style='path'")

### Pandas ↔ DuckDB

In [3]:
df = pd.DataFrame({"id": [1, 2, 3],"name": ["A", "B", "C"],"value": [10, 20, 30]})
con.register("df_tbl", df)
result = con.execute("SELECT name, value * 2 AS value2 FROM df_tbl WHERE value >= 20").df()
print(result)

  name  value2
0    B      40
1    C      60


### SQL 결과 → Pandas

In [4]:
out_df = con.execute("SELECT sum(value) AS total FROM df_tbl").df()
print(out_df)

   total
0   60.0


### S3 Parquet 파일 읽기

In [6]:
s3_url = "s3://mmix-prod-dataengineer-datalakehouse/streaming/mmix_mysql_cdc/mmix_departments_iceberg_sink/data/*.parquet"
dataframe = con.execute(f"""SELECT * FROM read_parquet('{s3_url}') LIMIT 100""").df()
dataframe

,id,name
0,10.0,조사 전문가
1,9.0,기타 문리/기술 및 예능 강사
2,8.0,통계관련 사무원
3,7.0,Legal
4,12.0,판금기 조작원
5,11.0,자동차 정비원
6,2.0,Design
7,1.0,Engineering
8,6.0,HR
9,5.0,Sales
